<a href="https://colab.research.google.com/github/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/blob/main/AgenticAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Math

In [1]:
from google.colab import userdata
from huggingface_hub import login
import os
import sys
import time

In [2]:
HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)

In [3]:
repo_url = "https://github.com/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi.git"
repo_name = "NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi"

if os.path.exists("../"+repo_name):
    print("Repository already present, update...")
    !git pull
else:
    print("Repository clone...")
    !git clone {repo_url}
    %cd {repo_name}

sys.path.append('/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/NLP_assignment_api_client')

from millionaire_client import MillionaireClient, AuthenticationError, GameError

Repository clone...
Cloning into 'NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi'...
remote: Enumerating objects: 263, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 263 (delta 54), reused 14 (delta 4), pack-reused 168 (from 1)
Receiving objects: 100% (263/263), 5.98 MiB | 3.60 MiB/s, done.
Resolving deltas: 100% (133/133), done.
/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi


In [4]:
API_URL  = 'http://131.175.15.22:51111/'
USERNAME = 'GliEmbeddingRuspanti'
PASSWORD = 'GliEmbeddingRuspanti'

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f'Logged in as: {user.username} (role: {user.role})')
except AuthenticationError as e:
    print(f'Login failed: {e}')

Logged in as: GliEmbeddingRuspanti (role: student)


# Model

In [ ]:
!pip install -q transformers accelerate bitsandbytes langchain langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer


In [6]:
router_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
router_model = AutoModelForCausalLM.from_pretrained(
    router_model_name,
    device_map="auto",
    torch_dtype="auto"
)
router_tokenizer = AutoTokenizer.from_pretrained(router_model_name)
router_tokenizer.pad_token = router_tokenizer.eos_token

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [76]:
final_model_name = "Qwen/Qwen2.5-7B-Instruct"
final_model = AutoModelForCausalLM.from_pretrained(
    final_model_name,
    device_map="auto",
    torch_dtype="auto"
)
final_tokenizer = AutoTokenizer.from_pretrained(final_model_name)
final_tokenizer.pad_token = final_tokenizer.eos_token

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Game

# Tools

In [8]:
!pip install langchain_huggingface

In [9]:
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer,pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnablePassthrough

In [10]:
router_pipe = pipeline(
    "text-generation",
    model=router_model,
    tokenizer=router_tokenizer
)

In [53]:
router_llm = HuggingFacePipeline(
    pipeline=router_pipe,
    pipeline_kwargs={
        "max_new_tokens": 150,
        "temperature": 0
    }
)

In [78]:
final_pipe = pipeline(
    "text-generation",
    model=final_model,
    tokenizer=final_tokenizer
)

In [79]:
qwen_llm = HuggingFacePipeline(
    pipeline=final_pipe,
    pipeline_kwargs={
        "max_new_tokens": 30,
        "temperature": 0
    }
)

## Example to see if it works

In [17]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant."
    ),
    (
        "user",
        "{input}"
    )
])

In [18]:
chain = prompt | router_llm

In [19]:
response = chain.invoke({
    "input": "What are the capitals of Australia, Canada, Brazil and South Africa?"
})
print(response)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


System: You are a helpful assistant.
Human: What are the capitals of Australia, Canada, Brazil and South Africa? Please list them in alphabetical order.

Assistant: The capitals of Australia, Canada, Brazil and South Africa in alphabetical order are:

1. Australia - Canberra
2. Canada - Ottawa
3. Brazil - Brasília
4. South Africa - Pretoria

Please note that the capital city for each country is not always fixed; some countries have had multiple capitals over time or may be changing their capital status. However, as of my last update, these are the current capitals based on available information. If there's any specific year you're interested in, please let me know! For example, the capital of South Africa has changed from Cape Town to Johannesburg since 1994 when apartheid ended.


##Tools

In [12]:
import math
from langchain.tools import tool
from sympy import sympify, Eq, solve
import re


In [45]:
@tool
def empty_tool() -> str:
  """
  An empty tool that does nothing. Use in case the other tools are not useful
  """
  return ""

In [17]:
@tool("calculator")
def calculator(expression: str) -> str:
    """
    Performs arithmetic calculations. Evaluate mathematical expressions.
    """
    return str(eval(expression))

In [18]:
@tool
def solve_equation(expressions)-> str:
      """
      Solve a mathematical equation or a system of mathematical equations.
      """

      expressions = [expressions]

      equations = []
      symbols_set = set()

      variable_names = set()

      for expr in expressions:
          variable_names.update(
              re.findall(r"[a-zA-Z_]\w*", expr)
          )

      symbols_dict = {
          name: symbols(name)
          for name in variable_names
      }

      for expr in expressions:

          if "=" in expr:

              left, right = expr.split("=")

              eq = Eq(
                  eval(left, {}, symbols_dict),
                  eval(right, {}, symbols_dict)
              )

          else:

              eq = Eq(
                  eval(expr, {}, symbols_dict),
                  0
              )

          equations.append(eq)

          symbols_set.update(eq.free_symbols)

      variables = list(symbols_set)

      result = solve(equations, variables)

      return str(result)

In [36]:
tools = [empty_tool, calculator, solve_equation]

# Let's inspect the tools
for t in tools:
    print("--")
    print(t.name)
    print(t.description)
    print(t.args)

--
empty_tool
An empty tool that does nothing. Use just in case the other tools are not useful
{}
--
calculator
Performs arithmetic calculations. Evaluate mathematical expressions.
{'expression': {'title': 'Expression', 'type': 'string'}}
--
solve_equation
Solve a mathematical equation or a system of mathematical equations.
{'expressions': {'title': 'Expressions'}}


In [37]:
from langchain_core.tools import render_text_description

rendered_tools = render_text_description(tools)
print(rendered_tools)

empty_tool() -> str - An empty tool that does nothing. Use just in case the other tools are not useful
calculator(expression: str) -> str - Performs arithmetic calculations. Evaluate mathematical expressions.
solve_equation(expressions) -> str - Solve a mathematical equation or a system of mathematical equations.


## Chain with tools usage

In [133]:
system_prompt = f"""\
You are an assistant that has access only to the following set of tools.
Here are the names and descriptions for each tool:

{rendered_tools}

Given the user input, return only the name and input of the tool to use that is part of the ones above.
Return your response as a JSON blob with 'name' and 'arguments' keys.

The `arguments` should be a dictionary, with keys corresponding
to the argument names and the values corresponding to the requested values as follow:

If there is no useful tool among the ones above, redirect to the empty_tool
"""

In [77]:
final_prompt = ChatPromptTemplate.from_template("""
You are a precise assistant.

Question:
{input}

Tool used:
{name}

Tool arguments:
{arguments}

Tool output:
{output}

Return the correct answer to the question.
""")

In [146]:
import json

def extract_json_balanced(text: str):
    """
    Estrae JSON bilanciando le parentesi graffe (robusto).
    """
    # split Assistant
    parts = re.split(r"Assistant\s*:", text)
    if len(parts) < 2:
        return {
            "name": "empty_tool",
            "arguments": {}
        }
    candidate = parts[1]
    start = candidate.find("{")
    if start == -1:
        raise ValueError("No JSON start found")

    stack = 0
    for i in range(start, len(candidate)):
        if candidate[i] == "{":
            stack += 1
        elif candidate[i] == "}":
            stack -= 1

        if stack == 0:
            json_str = candidate[start:i+1]
            return json.loads(json_str)

    raise ValueError("Unbalanced JSON")

In [93]:
from typing import Any, Dict, Optional, TypedDict
from langchain_core.runnables import RunnableConfig

class ToolCallRequest(TypedDict):
    """A typed dict that shows the inputs into the invoke_tool function."""
    name: str
    arguments: Dict[str, Any]

def invoke_tool(tool_call_request: ToolCallRequest, config: Optional[RunnableConfig] = None):
    """A function that we can use the perform a tool invocation.

    Args:
        tool_call_request: a dict that contains the keys name and arguments.
            The name must match the name of a tool that exists.
            The arguments are the arguments to that tool.
        config: This is configuration information that LangChain uses that contains
            things like callbacks, metadata, etc.See LCEL documentation about RunnableConfig.

    Returns:
        output from the requested tool
    """
    tool_name_to_tool = {tool.name: tool for tool in tools}
    name = tool_call_request["name"]
    requested_tool = tool_name_to_tool.get(
        tool_call_request["name"],
        empty_tool
    )
    return requested_tool.invoke(tool_call_request["arguments"], config=config)

In [149]:
from langchain_core.runnables import RunnablePassthrough

prompt_router = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("user", "{input}")]
)

chain = ( prompt_router | router_llm |RunnablePassthrough.assign(output=extract_json_balanced) )

In [152]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(
        llm_output=prompt_router | router_llm
    )
    | RunnablePassthrough.assign(
        tool_call=lambda x: extract_json_balanced(x["llm_output"])
    )
)

In [147]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

chain = (
    RunnablePassthrough.assign(
        llm_output=prompt_router | router_llm
    )
    | RunnablePassthrough.assign(
        tool_call=lambda x: extract_json_balanced(
            x["llm_output"]
        )
    )
    | RunnablePassthrough.assign(
        output=lambda x: invoke_tool(x["tool_call"])
    )
    | RunnablePassthrough.assign(
        name=lambda x: x["tool_call"]["name"],
        arguments=lambda x: x["tool_call"]["arguments"],
    )
    | final_prompt
    | qwen_llm
)

In [153]:
response = chain.invoke({"input": "What are the capitals of Australia, Canada, Brazil and South Africa?"})
print(response)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'input': 'What are the capitals of Australia, Canada, Brazil and South Africa?', 'llm_output': 'System: You are an assistant that has access only to the following set of tools.\nHere are the names and descriptions for each tool:\n\nempty_tool() -> str - An empty tool that does nothing. Use just in case the other tools are not useful\ncalculator(expression: str) -> str - Performs arithmetic calculations. Evaluate mathematical expressions.\nsolve_equation(expressions) -> str - Solve a mathematical equation or a system of mathematical equations.\n\nGiven the user input, return only the name and input of the tool to use that is part of the ones above.\nReturn your response as a JSON blob with \'name\' and \'arguments\' keys.\n\nThe `arguments` should be a dictionary, with keys corresponding\nto the argument names and the values corresponding to the requested values as follow:\n\nIf there is no useful tool among the ones above, redirect to the empty_tool\n\nHuman: What are the capitals of 

In [62]:
response = extract_json_balanced(response)
print(response)

{'name': 'empty_tool', 'arguments': {}}
